# EE 451: Communications Systems
## Lesson 23 — Probability & Channel Capacity

### Learning Objectives
By the end of this lesson, you will be able to:
- Apply conditional probability and Bayes' theorem to communication problems
- Explain Shannon's channel capacity theorem
- Calculate channel capacity for AWGN channels
- Relate SNR, bandwidth, and data rate through Shannon's theorem
- Understand fundamental limits of communication systems

### Textbook Reference
Haykin & Moher, Chapter 8.1–8.2

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: Shannon-Hartley Theorem

**Shannon's fundamental result (1948):**

$$C = B \log_2(1 + \text{SNR}) \;\; \text{bits/sec}$$

where:
- $C$ = channel capacity (maximum achievable data rate with arbitrarily low error)
- $B$ = channel bandwidth (Hz)
- $\text{SNR} = P_{signal} / P_{noise}$ (linear, not dB)

**Shannon's Noisy Channel Coding Theorem:**
- If $R < C$: Error-free communication is possible (with sufficiently good coding)
- If $R > C$: Reliable communication is impossible, regardless of coding

In [ ]:
# === Part 1: Shannon-Hartley Theorem ===

# Capacity vs SNR for fixed bandwidth
B = 10e6  # 10 MHz bandwidth
SNR_dB = np.linspace(-5, 40, 200)
SNR_lin = 10**(SNR_dB / 10)
C = B * np.log2(1 + SNR_lin)  # bits/sec

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# C vs SNR
axes[0].plot(SNR_dB, C / 1e6, 'C0', linewidth=2.5)
axes[0].set_xlabel('SNR (dB)', fontsize=13)
axes[0].set_ylabel('Channel Capacity (Mbps)', fontsize=13)
axes[0].set_title(f'Capacity vs SNR (B = {B/1e6:.0f} MHz)', fontsize=14, fontweight='bold')

# Mark some practical points
for snr_val, label in [(5, 'Cell edge'), (15, 'Indoor'), (25, 'Near AP'), (35, 'Line-of-sight')]:
    snr_lin = 10**(snr_val / 10)
    cap = B * np.log2(1 + snr_lin) / 1e6
    axes[0].plot(snr_val, cap, 'ko', markersize=6)
    axes[0].annotate(f'{label}\n{cap:.0f} Mbps', xy=(snr_val, cap),
                     xytext=(snr_val + 2, cap + 10), fontsize=9,
                     arrowprops=dict(arrowstyle='->', color='gray'))

# C vs B for fixed SNR
bandwidths = np.linspace(0.1e6, 100e6, 200)
for snr_fixed_dB, color in [(5, 'C1'), (10, 'C2'), (15, 'C0'), (20, 'C3')]:
    snr_fixed = 10**(snr_fixed_dB / 10)
    C_bw = bandwidths * np.log2(1 + snr_fixed)
    axes[1].plot(bandwidths / 1e6, C_bw / 1e6, color, linewidth=2,
                label=f'SNR = {snr_fixed_dB} dB')

axes[1].set_xlabel('Bandwidth (MHz)', fontsize=13)
axes[1].set_ylabel('Channel Capacity (Mbps)', fontsize=13)
axes[1].set_title('Capacity vs Bandwidth', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Worked examples
print("Shannon-Hartley Worked Examples:")
print("=" * 55)
examples = [
    ('Telephone', 4e3, 30),
    ('WiFi 20 MHz', 20e6, 25),
    ('LTE 10 MHz', 10e6, 15),
    ('Satellite', 36e6, 10),
]
for name, bw, snr in examples:
    snr_lin = 10**(snr / 10)
    cap = bw * np.log2(1 + snr_lin)
    eta = np.log2(1 + snr_lin)
    print(f"  {name:.<20s} B={bw/1e6:>5.1f} MHz, SNR={snr:>2d} dB "
          f"\u2192 C = {cap/1e6:>7.1f} Mbps (\u03b7 = {eta:.2f} b/s/Hz)")

## Part 2: Spectral Efficiency Limits

**Spectral efficiency** measures how efficiently bandwidth is used:

$$\eta = \frac{C}{B} = \log_2(1 + \text{SNR}) \;\; \text{bits/s/Hz}$$

No real system can exceed the Shannon limit. Modern systems with
advanced coding (LDPC, Turbo, Polar codes) operate within 1–2 dB of it.

In [ ]:
# === Part 2: Spectral Efficiency — Theory vs Practice ===

SNR_dB = np.linspace(-5, 35, 200)
SNR_lin = 10**(SNR_dB / 10)
eta_shannon = np.log2(1 + SNR_lin)

fig, ax = plt.subplots(figsize=(11, 7))

# Shannon limit
ax.plot(SNR_dB, eta_shannon, 'k-', linewidth=3, label='Shannon Limit', zorder=5)
ax.fill_between(SNR_dB, 0, eta_shannon, alpha=0.08, color='green')
ax.text(20, 1.5, 'Achievable\nRegion', fontsize=14, color='green', fontweight='bold', alpha=0.6)
ax.text(5, 6, 'Impossible\nRegion', fontsize=14, color='red', fontweight='bold', alpha=0.6)

# Real systems
systems = [
    ('BPSK', 0, 10.5, 1.0, 'v', 'C0'),
    ('QPSK', 2, 10.5, 2.0, 's', 'C1'),
    ('16-QAM', 4, 14.5, 4.0, 'D', 'C2'),
    ('64-QAM', 6, 20, 6.0, '^', 'C3'),
    ('256-QAM', 8, 26, 8.0, 'o', 'C4'),
    ('WiFi 6 (max)', 9.6, 30, 9.6, '*', 'C5'),
]

for name, eta, snr, rate, marker, color in systems:
    ax.plot(snr, eta, marker, markersize=12, color=color, markeredgecolor='black',
            markeredgewidth=0.5, label=f'{name} (\u03b7={eta} b/s/Hz)', zorder=6)

# Shannon limit at -1.59 dB
ax.axvline(-1.59, color='red', linestyle=':', alpha=0.6)
ax.text(-1.3, 7, 'Shannon\nlimit\n-1.59 dB', fontsize=9, color='red')

ax.set_xlabel('SNR per bit, $E_b/N_0$ (dB)', fontsize=13)
ax.set_ylabel('Spectral Efficiency (bits/s/Hz)', fontsize=13)
ax.set_title('Spectral Efficiency: Shannon Limit vs Real Modulation Schemes',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.set_xlim(-5, 35)
ax.set_ylim(0, 12)

plt.tight_layout()
plt.show()

print("Spectral Efficiency Comparison:")
print(f"{'Modulation':<18} {'\u03b7 (b/s/Hz)':<14} {'SNR Required':<15} {'Gap to Shannon'}")
print("-" * 60)
for name, eta, snr, rate, _, _ in systems:
    # Shannon requires this SNR for the same eta
    snr_shannon = 10 * np.log10(2**eta - 1)
    gap = snr - snr_shannon
    print(f"{name:<18} {eta:<14.1f} {snr:<15.1f} {gap:>8.1f} dB")

## Part 3: Bandwidth–SNR Trade-off

For a fixed data rate $R$, we can trade bandwidth for SNR:
- **More bandwidth** $\Rightarrow$ lower SNR required (spread spectrum regime)
- **Less bandwidth** $\Rightarrow$ higher SNR required (bandwidth-limited regime)

$$R = B \log_2(1 + \text{SNR}) \quad \Longrightarrow \quad \text{SNR} = 2^{R/B} - 1$$

In [ ]:
# === Part 3: Bandwidth-SNR Trade-off ===

R_target = 100e6  # Target: 100 Mbps

bandwidths = np.linspace(5e6, 200e6, 200)
SNR_required = 2**(R_target / bandwidths) - 1
SNR_required_dB = 10 * np.log10(SNR_required)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SNR vs Bandwidth
axes[0].plot(bandwidths / 1e6, SNR_required_dB, 'C0', linewidth=2.5)
axes[0].set_xlabel('Bandwidth (MHz)', fontsize=13)
axes[0].set_ylabel('Required SNR (dB)', fontsize=13)
axes[0].set_title(f'Required SNR for R = {R_target/1e6:.0f} Mbps', fontsize=14, fontweight='bold')

# Mark two design options from lesson plan
options = [(20, 'Option 1'), (40, 'Option 2'), (100, 'Option 3')]
for bw_mhz, label in options:
    snr_req = 2**(R_target / (bw_mhz * 1e6)) - 1
    snr_req_dB = 10 * np.log10(snr_req)
    axes[0].plot(bw_mhz, snr_req_dB, 'ro', markersize=10)
    axes[0].annotate(f'{label}\nB={bw_mhz} MHz\nSNR={snr_req_dB:.1f} dB',
                     xy=(bw_mhz, snr_req_dB), xytext=(bw_mhz + 15, snr_req_dB + 3),
                     fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

# Contour plot: C as function of B and SNR
B_range = np.linspace(1e6, 100e6, 100)
SNR_range_dB = np.linspace(0, 30, 100)
B_grid, SNR_grid = np.meshgrid(B_range, SNR_range_dB)
SNR_grid_lin = 10**(SNR_grid / 10)
C_grid = B_grid * np.log2(1 + SNR_grid_lin) / 1e6  # Mbps

levels = [10, 25, 50, 100, 200, 500, 1000]
cs = axes[1].contour(B_grid / 1e6, SNR_grid, C_grid, levels=levels, cmap='viridis')
axes[1].clabel(cs, fmt='%d Mbps', fontsize=9)
axes[1].set_xlabel('Bandwidth (MHz)', fontsize=13)
axes[1].set_ylabel('SNR (dB)', fontsize=13)
axes[1].set_title('Channel Capacity Contours (Mbps)', fontsize=14, fontweight='bold')

# Mark real systems on contour
real_sys = [
    ('WiFi', 20, 25, 'C0'),
    ('LTE', 10, 15, 'C1'),
    ('5G', 100, 20, 'C3'),
]
for name, bw, snr, color in real_sys:
    axes[1].plot(bw, snr, 'o', color=color, markersize=10, markeredgecolor='black')
    axes[1].text(bw + 2, snr + 1, name, fontsize=10, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

print(f"Design Trade-off for R = {R_target/1e6:.0f} Mbps:")
print(f"{'Option':<12} {'Bandwidth':<15} {'Required SNR':<18} {'Spectral Eff.'}")
print("-" * 60)
for bw_mhz, label in options:
    snr_req = 2**(R_target / (bw_mhz * 1e6)) - 1
    snr_req_dB = 10 * np.log10(snr_req)
    eta = R_target / (bw_mhz * 1e6)
    print(f"{label:<12} {bw_mhz:>6} MHz       {snr_req_dB:>8.1f} dB        {eta:.2f} b/s/Hz")
print(f"\nDoubling bandwidth from 20 to 40 MHz saves "
      f"{10*np.log10(2**(100/20)-1) - 10*np.log10(2**(100/40)-1):.1f} dB in SNR requirement")

## Part 4: Practical Systems vs Shannon Limit

Modern coding techniques have brought real systems remarkably close to
Shannon's theoretical limit:

| Code | Year | Gap to Shannon | Used In |
|------|------|---------------|--------|
| Hamming | 1950 | ~6 dB | Early systems |
| Reed-Solomon | 1960 | ~3 dB | CDs, DVDs, QR codes |
| Convolutional + Viterbi | 1967 | ~2 dB | Deep space, GSM |
| **Turbo codes** | 1993 | **~0.5 dB** | 3G/4G, deep space |
| **LDPC** | 1996 | **~0.3 dB** | WiFi, DVB-S2, 5G |
| **Polar codes** | 2009 | **~0.1 dB** | 5G NR control |

In [ ]:
# === Part 4: Approaching the Shannon Limit ===

# Shannon limit curve: minimum Eb/N0 for a given spectral efficiency
eta_range = np.linspace(0.01, 10, 500)
# Shannon: C/B = log2(1 + Eb/N0 * C/B)
# Eb/N0 = (2^eta - 1) / eta
EbN0_shannon = (2**eta_range - 1) / eta_range
EbN0_shannon_dB = 10 * np.log10(EbN0_shannon)

fig, ax = plt.subplots(figsize=(11, 7))

# Shannon limit
ax.plot(EbN0_shannon_dB, eta_range, 'k-', linewidth=3, label='Shannon Limit')
ax.fill_betweenx(eta_range, EbN0_shannon_dB, 20, alpha=0.05, color='green')
ax.fill_betweenx(eta_range, -2, EbN0_shannon_dB, alpha=0.05, color='red')

# Practical systems with their achieved performance
practical = [
    ('Uncoded BPSK', 10.5, 1.0, 'D', 'C7', 10),
    ('Hamming (7,4)', 7.0, 0.57, 's', 'C1', 10),
    ('Conv. + Viterbi', 4.5, 1.0, '^', 'C2', 10),
    ('Turbo code', 1.1, 1.0, 'o', 'C3', 12),
    ('LDPC (WiFi 6)', 0.8, 5.0, 'p', 'C0', 12),
    ('5G Polar', 0.5, 1.0, '*', 'C4', 14),
    ('WiFi 6 (1024-QAM)', 2.5, 8.5, 'h', 'C5', 12),
    ('LTE (64-QAM)', 2.0, 4.5, 'X', 'C6', 12),
]

for name, ebn0, eta, marker, color, ms in practical:
    ax.plot(ebn0, eta, marker, color=color, markersize=ms, markeredgecolor='black',
            markeredgewidth=0.5, label=name, zorder=6)

# Ultimate Shannon limit
ax.axvline(-1.59, color='red', linestyle=':', linewidth=2, alpha=0.6)
ax.text(-1.4, 8, 'Eb/N0 = -1.59 dB\n(Shannon limit as \u03b7 \u2192 0)',
        fontsize=10, color='red')

ax.set_xlabel('$E_b/N_0$ (dB)', fontsize=13)
ax.set_ylabel('Spectral Efficiency (bits/s/Hz)', fontsize=13)
ax.set_title('Practical Systems Approaching Shannon Limit', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.set_xlim(-2, 15)
ax.set_ylim(0, 10)

plt.tight_layout()
plt.show()

print("Progress Toward Shannon Limit:")
print(f"{'Coding Technique':<25} {'Year':<8} {'Gap to Shannon':<18} {'Application'}")
print("-" * 75)
timeline = [
    ('Uncoded BPSK', '1950s', '~10 dB', 'Basic digital'),
    ('Hamming codes', '1950', '~6 dB', 'Early computers'),
    ('Conv. + Viterbi', '1967', '~2 dB', 'Deep space, GSM'),
    ('Turbo codes', '1993', '~0.5 dB', '3G/4G cellular'),
    ('LDPC codes', '1996', '~0.3 dB', 'WiFi, DVB-S2, 5G'),
    ('Polar codes', '2009', '~0.1 dB', '5G NR control'),
]
for tech, year, gap, app in timeline:
    print(f"{tech:<25} {year:<8} {gap:<18} {app}")

## Part 5: System Design Example

**Problem:** Design a wireless link to achieve 100 Mbps.

Given Shannon's theorem, we can choose different bandwidth/SNR combinations.
The trade-off determines the system architecture, modulation order, and coding.

In [ ]:
# === Part 5: System Design Trade-off ===

# Compare WiFi, LTE, and Satellite approaches to 100 Mbps
designs = {
    'WiFi (indoor)': {'B_MHz': 20, 'SNR_dB': 25, 'mod': '256-QAM 5/6'},
    'LTE (urban)': {'B_MHz': 20, 'SNR_dB': 15, 'mod': '64-QAM 3/4 + MIMO'},
    '5G mmWave': {'B_MHz': 100, 'SNR_dB': 15, 'mod': '64-QAM 3/4'},
    'Satellite': {'B_MHz': 36, 'SNR_dB': 10, 'mod': '8PSK + LDPC'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: capacity of each system
names = list(designs.keys())
capacities = []
achieved = [100, 75, 500, 72]  # Approximate real throughput (Mbps)

for name, params in designs.items():
    snr_lin = 10**(params['SNR_dB'] / 10)
    cap = params['B_MHz'] * np.log2(1 + snr_lin)
    capacities.append(cap)

x = np.arange(len(names))
w = 0.35
bars1 = axes[0].bar(x - w/2, capacities, w, label='Shannon Capacity', color='C0', alpha=0.7)
bars2 = axes[0].bar(x + w/2, achieved, w, label='Actual Throughput', color='C1', alpha=0.7)

# Show efficiency percentage
for i, (cap, ach) in enumerate(zip(capacities, achieved)):
    pct = ach / cap * 100
    axes[0].text(i, max(cap, ach) + 15, f'{pct:.0f}%', ha='center', fontsize=10, fontweight='bold')

axes[0].set_xticks(x)
axes[0].set_xticklabels(names, fontsize=10)
axes[0].set_ylabel('Rate (Mbps)', fontsize=13)
axes[0].set_title('Shannon Capacity vs Actual Throughput', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# Spectral efficiency bar chart
eta_shannon = [c / d['B_MHz'] for c, d in zip(capacities, designs.values())]
eta_actual = [a / d['B_MHz'] for a, d in zip(achieved, designs.values())]

bars3 = axes[1].bar(x - w/2, eta_shannon, w, label='Shannon \u03b7_max', color='C0', alpha=0.7)
bars4 = axes[1].bar(x + w/2, eta_actual, w, label='Actual \u03b7', color='C1', alpha=0.7)

axes[1].set_xticks(x)
axes[1].set_xticklabels(names, fontsize=10)
axes[1].set_ylabel('Spectral Efficiency (b/s/Hz)', fontsize=13)
axes[1].set_title('Spectral Efficiency Comparison', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("System Design Comparison:")
print(f"{'System':<18} {'BW (MHz)':<10} {'SNR (dB)':<10} {'C_Shannon':<12} {'Actual':<10} {'Efficiency'}")
print("-" * 72)
for i, (name, params) in enumerate(designs.items()):
    eff = achieved[i] / capacities[i] * 100
    print(f"{name:<18} {params['B_MHz']:<10} {params['SNR_dB']:<10} "
          f"{capacities[i]:>8.1f} Mbps {achieved[i]:>6} Mbps  {eff:>5.0f}%")

## Summary

### Key Formulas

| Quantity | Formula |
|----------|--------|
| Shannon-Hartley capacity | $C = B \log_2(1 + \text{SNR})$ bits/sec |
| Spectral efficiency limit | $\eta_{\max} = C/B = \log_2(1 + \text{SNR})$ bits/s/Hz |
| Required SNR for rate $R$ | $\text{SNR} = 2^{R/B} - 1$ |
| Shannon limit ($\eta \to 0$) | $E_b/N_0 \geq \ln 2 = -1.59$ dB |

### Key Takeaways

1. **Shannon's theorem** sets an absolute limit on reliable communication rate
2. **Increasing bandwidth** $B$ increases capacity (linearly)
3. **Increasing SNR** increases capacity (logarithmically — diminishing returns)
4. **Bandwidth and SNR are tradeable** for a fixed target rate
5. **Modern codes** (LDPC, Turbo, Polar) approach within ~0.5 dB of Shannon limit
6. **Real systems** achieve 50–80% of Shannon capacity due to practical constraints

### Next Topics
- **Lesson 24:** Random Variables, PDFs, Gaussian Distribution
- Q-function and BER calculations
- Reading: Chapter 8.3–8.4